# 6 现代卷积神经网络

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import time

## 1. AlexNet

### 实现 AlexNet

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 96, kernel_size=11, stride=4, padding=1), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(256, 384, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(384, 384, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Flatten(),
            nn.Linear(6400, 4096), nn.ReLU(), nn.Dropout(p=0.5),
            nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(p=0.5),
            nn.Linear(4096, num_classes))

    def forward(self, x):
        return self.net(x)

In [ ]:
net = AlexNet()
X = torch.randn(1, 1, 224, 224)
for layer in net.net:
    X = layer(X)
    print(f'{layer.__class__.__name__:>10} -> {list(X.shape)}')

### 在 Fashion-MNIST 上训练 AlexNet

In [ ]:
def load_data_fashion_mnist(batch_size, resize=None):
    trans = [transforms.ToTensor()]
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)
    mnist_train = torchvision.datasets.FashionMNIST(
        root='../data', train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root='../data', train=False, transform=trans, download=True)
    return (DataLoader(mnist_train, batch_size, shuffle=True),
            DataLoader(mnist_test, batch_size, shuffle=False))


def evaluate_accuracy(net, data_iter, device):
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            correct += (net(X).argmax(dim=1) == y).sum().item()
            total += y.numel()
    return correct / total


def train(net, train_iter, test_iter, lr, num_epochs, device):
    def init_weights(m):
        if type(m) in (nn.Linear, nn.Conv2d):
            nn.init.xavier_uniform_(m.weight)
    net.apply(init_weights)
    net.to(device)
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()
    for epoch in range(num_epochs):
        net.train()
        total_loss, correct, total = 0.0, 0, 0
        for X, y in train_iter:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            l = loss(net(X), y)
            l.backward()
            optimizer.step()
            total_loss += l.item() * y.numel()
            correct += (net(X).argmax(dim=1) == y).sum().item()
            total += y.numel()
        train_acc = correct / total
        test_acc = evaluate_accuracy(net, test_iter, device)
        print(f'epoch {epoch + 1}: loss {total_loss / total:.4f}, '
              f'train acc {train_acc:.4f}, test acc {test_acc:.4f}')
    print(f'test acc {test_acc:.4f}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using {device}')
batch_size = 128
train_iter, test_iter = load_data_fashion_mnist(batch_size, resize=224)
net = AlexNet()
train(net, train_iter, test_iter, lr=0.01, num_epochs=2, device=device)

## 2. VGG

### VGG 块

In [ ]:
def vgg_block(num_convs, in_channels, out_channels):
    layers = []
    for _ in range(num_convs):
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
        layers.append(nn.ReLU())
        in_channels = out_channels
    layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
    return nn.Sequential(*layers)

### VGG-11 架构

In [ ]:
def vgg(conv_arch):
    conv_blks = []
    in_channels = 1
    for num_convs, out_channels in conv_arch:
        conv_blks.append(vgg_block(num_convs, in_channels, out_channels))
        in_channels = out_channels
    return nn.Sequential(
        *conv_blks,
        nn.Flatten(),
        nn.Linear(out_channels * 7 * 7, 4096), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(4096, 10))


conv_arch = ((1, 64), (1, 128), (2, 256), (2, 512), (2, 512))
net_vgg = vgg(conv_arch)

In [ ]:
X = torch.randn(1, 1, 224, 224)
for blk in net_vgg:
    X = blk(X)
    print(f'{blk.__class__.__name__:>10} -> {list(X.shape)}')

### 在 Fashion-MNIST 上训练 VGG-11

In [ ]:
train_iter, test_iter = load_data_fashion_mnist(128, resize=224)
net_vgg = vgg(conv_arch)
train(net_vgg, train_iter, test_iter, lr=0.01, num_epochs=2, device=device)

## 3. 网络中的网络（NiN）

### NiN 块

In [ ]:
def nin_block(in_channels, out_channels, kernel_size, strides, padding):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, strides, padding), nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1), nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1), nn.ReLU())

In [ ]:
class NiN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nin_block(1, 96, kernel_size=11, strides=4, padding=0),
            nn.MaxPool2d(3, stride=2),
            nin_block(96, 256, kernel_size=5, strides=1, padding=2),
            nn.MaxPool2d(3, stride=2),
            nin_block(256, 384, kernel_size=3, strides=1, padding=1),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout(0.5),
            nin_block(384, num_classes, kernel_size=3, strides=1, padding=1),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten())

    def forward(self, x):
        return self.net(x)

In [ ]:
net_nin = NiN()
X = torch.randn(1, 1, 224, 224)
for blk in net_nin.net:
    X = blk(X)
    print(f'{blk.__class__.__name__:>15} -> {list(X.shape)}')

In [ ]:
train_iter, test_iter = load_data_fashion_mnist(128, resize=224)
net_nin = NiN()
train(net_nin, train_iter, test_iter, lr=0.01, num_epochs=2, device=device)

## 4. GoogLeNet

### Inception 块

In [ ]:
class Inception(nn.Module):
    def __init__(self, in_channels, c1, c2, c3, c4):
        super().__init__()
        self.p1_1 = nn.Conv2d(in_channels, c1, kernel_size=1)
        self.p2_1 = nn.Conv2d(in_channels, c2[0], kernel_size=1)
        self.p2_2 = nn.Conv2d(c2[0], c2[1], kernel_size=3, padding=1)
        self.p3_1 = nn.Conv2d(in_channels, c3[0], kernel_size=1)
        self.p3_2 = nn.Conv2d(c3[0], c3[1], kernel_size=5, padding=2)
        self.p4_1 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.p4_2 = nn.Conv2d(in_channels, c4, kernel_size=1)

    def forward(self, x):
        p1 = torch.relu(self.p1_1(x))
        p2 = torch.relu(self.p2_2(torch.relu(self.p2_1(x))))
        p3 = torch.relu(self.p3_2(torch.relu(self.p3_1(x))))
        p4 = torch.relu(self.p4_2(self.p4_1(x)))
        return torch.cat((p1, p2, p3, p4), dim=1)

In [ ]:
class GoogLeNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.b1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
        self.b2 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=1), nn.ReLU(),
            nn.Conv2d(64, 192, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
        self.b3 = nn.Sequential(
            Inception(192, 64, (96, 128), (16, 32), 32),
            Inception(256, 128, (128, 192), (32, 96), 64),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
        self.b4 = nn.Sequential(
            Inception(480, 192, (96, 208), (16, 48), 64),
            Inception(512, 160, (112, 224), (24, 64), 64),
            Inception(512, 128, (128, 256), (24, 64), 64),
            Inception(512, 112, (144, 288), (32, 64), 64),
            Inception(528, 256, (160, 320), (32, 128), 128),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
        self.b5 = nn.Sequential(
            Inception(832, 256, (160, 320), (32, 128), 128),
            Inception(832, 384, (192, 384), (48, 128), 128),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(1024, num_classes))

    def forward(self, x):
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.b4(x)
        x = self.b5(x)
        return x

In [ ]:
net_googlenet = GoogLeNet()
X = torch.randn(1, 1, 96, 96)
for name, blk in net_googlenet.named_children():
    X = blk(X)
    print(f'{name:>5} -> {list(X.shape)}')

In [ ]:
train_iter, test_iter = load_data_fashion_mnist(128, resize=96)
net_googlenet = GoogLeNet()
train(net_googlenet, train_iter, test_iter, lr=0.01, num_epochs=2, device=device)

## 5. 批量规范化

### 使用 BatchNorm 改进 LeNet

In [ ]:
class LeNetBN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.BatchNorm2d(6), nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6, 16, kernel_size=5), nn.BatchNorm2d(16), nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120), nn.BatchNorm1d(120), nn.Sigmoid(),
            nn.Linear(120, 84), nn.BatchNorm1d(84), nn.Sigmoid(),
            nn.Linear(84, num_classes))

    def forward(self, x):
        return self.net(x)

In [ ]:
net_bn = LeNetBN()
X = torch.randn(1, 1, 28, 28)
for layer in net_bn.net:
    X = layer(X)
    print(f'{layer.__class__.__name__:>12} -> {list(X.shape)}')

In [ ]:
train_iter, test_iter = load_data_fashion_mnist(256)
net_bn = LeNetBN()
train(net_bn, train_iter, test_iter, lr=0.1, num_epochs=2, device=device)

## 6. ResNet

### 残差块

In [ ]:
class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
            self.bn3 = nn.BatchNorm2d(out_channels)
        else:
            self.conv3 = None

    def forward(self, x):
        y = torch.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.conv3:
            x = self.bn3(self.conv3(x))
        return torch.relu(y + x)

In [ ]:
blk = Residual(3, 3)
X = torch.randn(4, 3, 6, 6)
Y = blk(X)
print(f'输入形状: {list(X.shape)}, 输出形状: {list(Y.shape)}')
blk2 = Residual(3, 6, use_1x1conv=True, stride=2)
Y2 = blk2(X)
print(f'输入形状: {list(X.shape)}, 输出形状: {list(Y2.shape)}')

### ResNet-18 模型

In [ ]:
def resnet_block(in_channels, out_channels, num_residuals, first_block=False):
    blks = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blks.append(Residual(in_channels, out_channels, use_1x1conv=True, stride=2))
        else:
            blks.append(Residual(out_channels, out_channels))
    return blks


class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.b1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
        self.b2 = nn.Sequential(*resnet_block(64, 64, 2, first_block=True))
        self.b3 = nn.Sequential(*resnet_block(64, 128, 2))
        self.b4 = nn.Sequential(*resnet_block(128, 256, 2))
        self.b5 = nn.Sequential(*resnet_block(256, 512, 2))
        self.b6 = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Linear(512, num_classes))

    def forward(self, x):
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.b4(x)
        x = self.b5(x)
        x = self.b6(x)
        return x

In [ ]:
net_resnet = ResNet18()
X = torch.randn(1, 1, 224, 224)
for name, blk in net_resnet.named_children():
    X = blk(X)
    print(f'{name:>5} -> {list(X.shape)}')

### 在 Fashion-MNIST 上训练 ResNet-18

In [ ]:
train_iter, test_iter = load_data_fashion_mnist(128, resize=96)
net_resnet = ResNet18()
train(net_resnet, train_iter, test_iter, lr=0.01, num_epochs=2, device=device)

以上介绍了现代卷积神经网络中的代表模型：AlexNet、VGG、NiN、GoogLeNet（Inception）、批量规范化以及 ResNet。这些模型构成了现代深度学习计算机视觉的基础。